<a href="https://colab.research.google.com/github/AlperYildirim1/HAMON/blob/main/HAMON_Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
HAMON Ultimate Colab Script
===========================
Hardware-Accelerated Modulation Optical Network for time-series forecasting.

This version supports controlled ablations through four experiment flags:

  1. ENCODING_MODE:
       "amplitude" | "phase"

  2. READOUT_MODE:
       "coherent" | "intensity" | "differential_intensity"

  3. PHASE_ALPHA:
       Float phase scale for phase encoding.
       U_in = exp(i * PHASE_ALPHA * pi * x_norm)

  4. USE_BACKCAST_LOSS:
       True | False

Recommended ablation order:

  Baseline:
    ENCODING_MODE="amplitude"
    READOUT_MODE="coherent"
    USE_BACKCAST_LOSS=True

  Main phase-encoded invention candidate:
    ENCODING_MODE="phase"
    READOUT_MODE="coherent"
    PHASE_ALPHA=0.3
    USE_BACKCAST_LOSS=True

  Phase + ordinary intensity detector:
    ENCODING_MODE="phase"
    READOUT_MODE="intensity"
    PHASE_ALPHA=0.3
    USE_BACKCAST_LOSS=False

  Phase + differential intensity detector:
    ENCODING_MODE="phase"
    READOUT_MODE="differential_intensity"
    PHASE_ALPHA=0.3
    USE_BACKCAST_LOSS=False

Colab notes:
  - Mount Google Drive before running if you want persistent checkpoints.
  - Set USE_GOOGLE_DRIVE=True.
  - Change DATASETS_CONFIG and RUN_CONFIG at the bottom as needed.

Physics summary:
  - Input history can be encoded either as amplitude or phase.
  - Forecast positions are left dark.
  - Trainable phase masks + free-space propagation form the learned optical core.
  - The transfer-matrix trick is preserved because the optical core remains linear in U_in.
"""

# =============================================================================
# 0. IMPORTS & COLAB SETUP
# =============================================================================

import os
import math
import json
import random
import logging
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler


# =============================================================================
# 1. GLOBAL CONFIGURATION
# =============================================================================

USE_GOOGLE_DRIVE = True
DRIVE_MOUNT_PATH = "/content/drive"
BASE_DIR = "/content/drive/MyDrive/HAMON_Ultimate" if USE_GOOGLE_DRIVE else "/content/HAMON_Ultimate"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.exists(DRIVE_MOUNT_PATH):
            drive.mount(DRIVE_MOUNT_PATH)
        elif not os.path.exists(os.path.join(DRIVE_MOUNT_PATH, "MyDrive")):
            drive.mount(DRIVE_MOUNT_PATH)
    except Exception as exc:
        print(f"Google Drive mount skipped or failed: {exc}")
        BASE_DIR = "/content/HAMON_Ultimate"

SAVE_DIR = os.path.join(BASE_DIR, "checkpoints")
LOG_DIR = os.path.join(BASE_DIR, "logs")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
DATA_DIR = os.path.join(BASE_DIR, "data")

for d in [SAVE_DIR, LOG_DIR, RESULTS_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"BASE_DIR: {BASE_DIR}")
print(f"SAVE_DIR: {SAVE_DIR}")
print(f"LOG_DIR: {LOG_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"DATA_DIR: {DATA_DIR}")

# --- Dataset download base ---
HF_BASE = "https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/main/"

DATASETS_CONFIG = {
    "etth1": {
        "path": "ETT-small/ETTh1.csv",
        "split": [8640, 2880, 2880],
        "channels": 7,
        "batch_size": 128,
    },
    "etth2": {
        "path": "ETT-small/ETTh2.csv",
        "split": [8640, 2880, 2880],
        "channels": 7,
        "batch_size": 128,
    },
    "ettm1": {
        "path": "ETT-small/ETTm1.csv",
        "split": [34560, 11520, 11520],
        "channels": 7,
        "batch_size": 128,
    },
    "ettm2": {
        "path": "ETT-small/ETTm2.csv",
        "split": [34560, 11520, 11520],
        "channels": 7,
        "batch_size": 128,
    },
    "weather": {
        "path": "weather/weather.csv",
        "split": "70_10_20",
        "channels": 21,
        "batch_size": 64,
    },
    "exchange": {
        "path": "exchange_rate/exchange_rate.csv",
        "split": "70_10_20",
        "channels": 8,
        "batch_size": 128,
    },
    "electricity": {
        "path": "electricity/electricity.csv",
        "split": "70_10_20",
        "channels": 321,
        "batch_size": 64,
    },
    "traffic": {
        "path": "traffic/traffic.csv",
        "split": "70_10_20",
        "channels": 862,
        "batch_size": 16,
    },
}

# --- Core time-series settings ---
SEQ_LEN = 336
PRED_LENS = [96, 192, 336, 720]
LAYER_CANDIDATES = [16]
SEEDS = [1, 2, 3]

# --- Training hyperparameters ---
EPOCHS = 120
LR = 1e-2
WEIGHT_DECAY = 1e-3
PATIENCE = 15
GRAD_CLIP = 1.0
BF_LAMBDA = 0.5

# --- Optical hardware parameters ---
DEFAULT_GRID_SIZE = 1120
DIFFERENTIAL_GRID_SIZE = 2048
SPACING = 10e-6
WAVELENGTH = 1e-6
DISTANCE = 0.03

# =============================================================================
# 2. FOUR MAIN ABLATION FLAGS
# =============================================================================

# Flag 1: "amplitude" or "phase"
ENCODING_MODE = "phase"

# Flag 2: "coherent", "intensity", or "differential_intensity"
# Current ablation: phase encoding + differential intensity detection.
READOUT_MODE = "differential_intensity"

# Flag 3: phase scale for phase encoding. Ignored for amplitude encoding.
PHASE_ALPHA = 0.3

# Flag 4: backcast loss on/off.
# Intensity readouts are non-negative before differencing/calibration, so backcast is disabled here.
USE_BACKCAST_LOSS = False

# Optional tiny calibration for intensity readout.
# This is intentionally boundary-only, not a digital forecasting head.
USE_INTENSITY_AFFINE = True

# Grid policy.
# "auto" uses 2048 for differential intensity when needed.
GRID_POLICY = "auto"

# Runtime / resume controls.
# SHOW_PROGRESS=False is usually faster and cleaner in Colab.
SHOW_PROGRESS = False

# If a run is interrupted, rerunning the same script resumes from *__last.pt.
RESUME_EXISTING = True

# If the final best checkpoint already exists, skip retraining that seed/run.
# Keep False while actively experimenting; set True for large repeated sweeps.
SKIP_FINISHED_RUNS = False

# Device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


# =============================================================================
# 3. EXPERIMENT NAMING
# =============================================================================

def make_experiment_tag(
    encoding_mode=ENCODING_MODE,
    readout_mode=READOUT_MODE,
    phase_alpha=PHASE_ALPHA,
    use_backcast_loss=USE_BACKCAST_LOSS,
    use_intensity_affine=USE_INTENSITY_AFFINE,
):
    alpha_tag = f"a{phase_alpha:g}".replace(".", "p")
    bc_tag = "bc1" if use_backcast_loss else "bc0"
    aff_tag = "aff1" if use_intensity_affine else "aff0"
    if encoding_mode == "amplitude":
        return f"enc-amp__read-{readout_mode}__{bc_tag}__{aff_tag}"
    return f"enc-phase-{alpha_tag}__read-{readout_mode}__{bc_tag}__{aff_tag}"

# Add a run suffix so each ablation writes to a separate Drive folder.
# Current run: phase + differential intensity, L=16 broad sweep without Electricity/Traffic.
EXPERIMENT_TAG = make_experiment_tag() + "__no_elec_traffic_Hsel_L16_s123_120_epochs"
EXPERIMENT_SAVE_DIR = os.path.join(SAVE_DIR, EXPERIMENT_TAG)
EXPERIMENT_LOG_DIR = os.path.join(LOG_DIR, EXPERIMENT_TAG)
EXPERIMENT_RESULTS_DIR = os.path.join(RESULTS_DIR, EXPERIMENT_TAG)

for d in [EXPERIMENT_SAVE_DIR, EXPERIMENT_LOG_DIR, EXPERIMENT_RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Experiment tag: {EXPERIMENT_TAG}")
print(f"Experiment checkpoints: {EXPERIMENT_SAVE_DIR}")


# =============================================================================
# 4. VALIDATION OF FLAG COMBINATIONS
# =============================================================================

def validate_flags():
    valid_encodings = {"amplitude", "phase"}
    valid_readouts = {"coherent", "intensity", "differential_intensity"}

    if ENCODING_MODE not in valid_encodings:
        raise ValueError(f"ENCODING_MODE must be one of {valid_encodings}, got {ENCODING_MODE}")
    if READOUT_MODE not in valid_readouts:
        raise ValueError(f"READOUT_MODE must be one of {valid_readouts}, got {READOUT_MODE}")
    if PHASE_ALPHA <= 0:
        raise ValueError("PHASE_ALPHA must be positive.")

    if READOUT_MODE in {"intensity", "differential_intensity"} and USE_BACKCAST_LOSS:
        print("WARNING: intensity readout is non-negative. USE_BACKCAST_LOSS=True may be unstable.")
        print("Recommended: USE_BACKCAST_LOSS=False for intensity modes.")

    if ENCODING_MODE == "amplitude" and READOUT_MODE != "coherent":
        print("WARNING: amplitude + intensity modes are allowed, but not part of the main ablation plan.")

validate_flags()


# =============================================================================
# 5. REPRODUCIBILITY
# =============================================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =============================================================================
# 6. LOGGING
# =============================================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")


def get_logger(run_name):
    logger = logging.getLogger(run_name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    log_path = os.path.join(EXPERIMENT_LOG_DIR, f"{run_name}_{timestamp}.log")
    fh = logging.FileHandler(log_path)
    fh.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
    logger.addHandler(fh)

    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(ch)

    return logger


# =============================================================================
# 7. DATA DOWNLOAD / LOADING
# =============================================================================

def ensure_dataset_downloaded(dataset_name):
    cfg = DATASETS_CONFIG[dataset_name]
    rel_path = cfg["path"]
    local_path = os.path.join(DATA_DIR, rel_path)

    if os.path.exists(local_path):
        print(f"{dataset_name}: OK -> {local_path}")
        return local_path

    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    url = HF_BASE + rel_path
    print(f"Downloading {dataset_name} from HuggingFace...")
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    with open(local_path, "wb") as f:
        f.write(response.content)
    print(f"{dataset_name}: downloaded -> {local_path}")
    return local_path


class TSDataset(Dataset):
    def __init__(self, data, seq_len, pred_len):
        self.data = data
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.seq_len]
        y = self.data[idx + self.seq_len: idx + self.seq_len + self.pred_len]
        return torch.FloatTensor(x), torch.FloatTensor(y)


def load_data(dataset_name):
    cfg = DATASETS_CONFIG[dataset_name]
    local_path = ensure_dataset_downloaded(dataset_name)
    df = pd.read_csv(local_path)
    data = df.iloc[:, 1:].values

    if cfg["split"] == "70_10_20":
        n = len(data)
        train_len = int(n * 0.7)
        val_len = int(n * 0.1)
    else:
        train_len = cfg["split"][0]
        val_len = cfg["split"][1]

    scaler = StandardScaler()
    scaler.fit(data[:train_len])
    data = scaler.transform(data)

    train = data[:train_len]
    val = data[train_len: train_len + val_len]
    test = data[train_len + val_len:]
    return train, val, test, scaler, cfg


def make_loaders(train_data, val_data, test_data, pred_len, batch_size):
    train_loader = DataLoader(
        TSDataset(train_data, SEQ_LEN, pred_len),
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        TSDataset(val_data, SEQ_LEN, pred_len),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        TSDataset(test_data, SEQ_LEN, pred_len),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, val_loader, test_loader


# =============================================================================
# 8. MODEL COMPONENTS
# =============================================================================

class RevIN(nn.Module):
    def __init__(self, num_features, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.affine_weight = nn.Parameter(torch.ones(num_features))
        self.affine_bias = nn.Parameter(torch.zeros(num_features))
        self._mean = None
        self._std = None

    def forward(self, x, mode="norm"):
        if mode == "norm":
            self._mean = x.mean(dim=1, keepdim=True).detach()
            self._std = (x.var(dim=1, keepdim=True, unbiased=False) + self.eps).sqrt().detach()
            x = (x - self._mean) / self._std
            return x * self.affine_weight + self.affine_bias

        if mode == "denorm":
            if self._mean is None or self._std is None:
                raise RuntimeError("RevIN denorm called before norm.")
            x = (x - self.affine_bias) / (self.affine_weight + self.eps)
            return x * self._std + self._mean

        raise ValueError(f"Unknown RevIN mode: {mode}")


class HAMON(nn.Module):
    def __init__(
        self,
        lookback=336,
        horizon=96,
        channels=7,
        num_layers=16,
        grid_size=DEFAULT_GRID_SIZE,
        spacing=SPACING,
        wavelength=WAVELENGTH,
        distance=DISTANCE,
        encoding_mode=ENCODING_MODE,
        readout_mode=READOUT_MODE,
        phase_alpha=PHASE_ALPHA,
        use_intensity_affine=USE_INTENSITY_AFFINE,
    ):
        super().__init__()
        self.lookback = lookback
        self.horizon = horizon
        self.channels = channels
        self.grid_size = grid_size
        self.num_layers = num_layers
        self.encoding_mode = encoding_mode
        self.readout_mode = readout_mode
        self.phase_alpha = phase_alpha
        self.use_intensity_affine = use_intensity_affine

        if encoding_mode not in {"amplitude", "phase"}:
            raise ValueError(f"Unknown encoding_mode: {encoding_mode}")
        if readout_mode not in {"coherent", "intensity", "differential_intensity"}:
            raise ValueError(f"Unknown readout_mode: {readout_mode}")

        if readout_mode == "differential_intensity":
            total_len = lookback + 2 * horizon
        else:
            total_len = lookback + horizon

        assert total_len <= grid_size, (
            f"grid_size ({grid_size}) must be >= required optical span ({total_len}). "
            f"For differential_intensity, required span is lookback + 2*horizon."
        )

        self.total_len = total_len
        self.start_idx = (grid_size - total_len) // 2
        self.hist_start = self.start_idx
        self.hist_end = self.hist_start + lookback
        self.fc_start = self.hist_end
        self.fc_end = self.fc_start + horizon

        if readout_mode == "differential_intensity":
            self.fc_neg_start = self.fc_end
            self.fc_neg_end = self.fc_neg_start + horizon
        else:
            self.fc_neg_start = None
            self.fc_neg_end = None

        self.revin = RevIN(channels)

        # Trainable optical phase masks.
        self.phase_masks = nn.ParameterList([
            nn.Parameter(torch.zeros(grid_size)) for _ in range(num_layers)
        ])

        # Optional boundary calibration for non-negative intensity outputs.
        # Shape is channel-wise scalar. This is not a temporal forecasting head.
        if readout_mode in {"intensity", "differential_intensity"} and use_intensity_affine:
            self.readout_scale = nn.Parameter(torch.ones(channels))
            self.readout_bias = nn.Parameter(torch.zeros(channels))
        else:
            self.readout_scale = None
            self.readout_bias = None

        fx = torch.fft.fftfreq(grid_size, d=spacing)
        k = 2 * math.pi / wavelength
        evanescent_mask = (wavelength * fx) ** 2 <= 1.0
        phase_shift = k * distance * torch.sqrt(
            torch.clamp(1.0 - (wavelength * fx) ** 2, min=0.0)
        )
        H = torch.exp(1j * phase_shift) * evanescent_mask
        self.register_buffer("H", H.to(torch.complex64))

    def _propagate(self, U):
        U_f = torch.fft.fft(U, norm="ortho")
        U_f = U_f * self.H
        return torch.fft.ifft(U_f, norm="ortho")

    def _encode_input(self, x_flat):
        BxC, L = x_flat.shape
        U_in = torch.zeros(BxC, self.grid_size, dtype=torch.complex64, device=x_flat.device)

        if self.encoding_mode == "amplitude":
            U_in[:, self.hist_start:self.hist_end] = x_flat.to(torch.complex64)

        elif self.encoding_mode == "phase":
            phase = self.phase_alpha * math.pi * x_flat
            U_in[:, self.hist_start:self.hist_end] = torch.exp(1j * phase).to(torch.complex64)

        else:
            raise ValueError(f"Unknown encoding_mode: {self.encoding_mode}")

        return U_in

    def _optical_core_transfer_matrix(self, device):
        I = torch.eye(self.grid_size, dtype=torch.complex64, device=device)
        complex_masks = [torch.exp(1j * mask).to(torch.complex64) for mask in self.phase_masks]

        M = I
        for complex_mask in complex_masks:
            M = self._propagate(M)
            M = M * complex_mask
        M = self._propagate(M)
        return M

    def _readout(self, U, B, C):
        if self.readout_mode == "coherent":
            signal = torch.real(U)
            backcast_flat = signal[:, self.hist_start:self.hist_end]
            forecast_flat = signal[:, self.fc_start:self.fc_end]

        elif self.readout_mode == "intensity":
            signal = torch.abs(U) ** 2
            backcast_flat = signal[:, self.hist_start:self.hist_end]
            forecast_flat = signal[:, self.fc_start:self.fc_end]

        elif self.readout_mode == "differential_intensity":
            signal = torch.abs(U) ** 2
            backcast_flat = signal[:, self.hist_start:self.hist_end]
            pos = signal[:, self.fc_start:self.fc_end]
            neg = signal[:, self.fc_neg_start:self.fc_neg_end]
            forecast_flat = pos - neg

        else:
            raise ValueError(f"Unknown readout_mode: {self.readout_mode}")

        backcast = backcast_flat.reshape(B, C, self.lookback).permute(0, 2, 1)
        forecast = forecast_flat.reshape(B, C, self.horizon).permute(0, 2, 1)

        if self.readout_scale is not None and self.readout_bias is not None:
            forecast = forecast * self.readout_scale + self.readout_bias
            backcast = backcast * self.readout_scale + self.readout_bias

        return forecast, backcast

    def forward(self, x, return_backcast=False):
        B, L, C = x.shape
        if L != self.lookback:
            raise ValueError(f"Expected lookback {self.lookback}, got {L}")
        if C != self.channels:
            raise ValueError(f"Expected channels {self.channels}, got {C}")

        x_norm = self.revin(x, mode="norm")
        x_flat = x_norm.permute(0, 2, 1).reshape(B * C, L)

        U_in = self._encode_input(x_flat)
        M = self._optical_core_transfer_matrix(x.device)
        U = torch.matmul(U_in, M)

        forecast_norm, backcast = self._readout(U, B, C)
        forecast = self.revin(forecast_norm, mode="denorm")

        if return_backcast:
            return forecast, backcast, x_norm
        return forecast

    def optical_summary(self):
        n_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        n_optical = sum(p.numel() for p in self.phase_masks)
        n_boundary = n_params - n_optical
        device_len = (self.num_layers + 1) * DISTANCE * 100
        return {
            "total_params": n_params,
            "optical_params": n_optical,
            "boundary_params": n_boundary,
            "num_layers": self.num_layers,
            "device_length_cm": device_len,
            "grid_size": self.grid_size,
            "required_span": self.total_len,
            "encoding_mode": self.encoding_mode,
            "readout_mode": self.readout_mode,
            "phase_alpha": self.phase_alpha,
            "use_intensity_affine": self.use_intensity_affine,
            "spacing_um": SPACING * 1e6,
            "wavelength_um": WAVELENGTH * 1e6,
            "distance_cm": DISTANCE * 100,
        }


# =============================================================================
# 9. GRID SIZE POLICY
# =============================================================================

def choose_grid_size(pred_len):
    if GRID_POLICY != "auto":
        return int(GRID_POLICY)

    if READOUT_MODE == "differential_intensity":
        required = SEQ_LEN + 2 * pred_len
        if required <= DEFAULT_GRID_SIZE:
            return DEFAULT_GRID_SIZE
        if required <= DIFFERENTIAL_GRID_SIZE:
            return DIFFERENTIAL_GRID_SIZE
        # Next power of two for FFT efficiency.
        return 2 ** math.ceil(math.log2(required))

    required = SEQ_LEN + pred_len
    if required <= DEFAULT_GRID_SIZE:
        return DEFAULT_GRID_SIZE
    return 2 ** math.ceil(math.log2(required))


# =============================================================================
# 10. TRAINING AND EVALUATION
# =============================================================================

def evaluate(model, loader, criterion):
    model.eval()
    total_mse, total_mae, n = 0.0, 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            pred = model(x)
            bs = x.size(0)
            total_mse += criterion(pred, y).item() * bs
            total_mae += nn.functional.l1_loss(pred, y).item() * bs
            n += bs
    return total_mse / n, total_mae / n


def train_single(
    model,
    train_loader,
    val_loader,
    ckpt_path,
    logger,
    epochs=EPOCHS,
    patience=PATIENCE,
    use_backcast_loss=USE_BACKCAST_LOSS,
):
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.MSELoss()

    last_ckpt_path = ckpt_path.replace(".pt", "__last.pt")
    best_val_mse = float("inf")
    patience_counter = 0
    start_epoch = 1

    # Resume from the most recent epoch checkpoint if available.
    if RESUME_EXISTING and os.path.exists(last_ckpt_path):
        resume_ckpt = torch.load(last_ckpt_path, map_location=device)
        model.load_state_dict(resume_ckpt["model_state_dict"])
        optimizer.load_state_dict(resume_ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(resume_ckpt["scheduler_state_dict"])
        best_val_mse = resume_ckpt.get("best_val_mse", float("inf"))
        patience_counter = resume_ckpt.get("patience_counter", 0)
        start_epoch = resume_ckpt.get("epoch", 0) + 1
        logger.info(
            f"  RESUME | loaded {last_ckpt_path} | "
            f"next epoch={start_epoch} | best_val={best_val_mse:.6f} | "
            f"patience={patience_counter}/{patience}"
        )

    if SKIP_FINISHED_RUNS and os.path.exists(ckpt_path) and not os.path.exists(last_ckpt_path):
        logger.info(f"  SKIP | found finished checkpoint: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=device)
        return ckpt.get("best_val_mse", float("nan")) if isinstance(ckpt, dict) else float("nan")

    if start_epoch > epochs:
        logger.info(f"  DONE | last checkpoint already reached epoch {epochs}")
        return best_val_mse

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        train_fc_losses = []
        train_bc_losses = []
        train_total_losses = []

        iterator = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", leave=False) if SHOW_PROGRESS else train_loader
        for x, y in iterator:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            if use_backcast_loss:
                forecast, backcast, x_norm = model(x, return_backcast=True)
                loss_fc = criterion(forecast, y)
                loss_bc = criterion(backcast, x_norm)
                loss = loss_fc + BF_LAMBDA * loss_bc
            else:
                forecast = model(x)
                loss_fc = criterion(forecast, y)
                loss_bc = torch.tensor(0.0, device=x.device)
                loss = loss_fc

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            optimizer.step()

            train_fc_losses.append(loss_fc.item())
            train_bc_losses.append(loss_bc.item())
            train_total_losses.append(loss.item())

            if SHOW_PROGRESS:
                iterator.set_postfix({"fc": f"{loss_fc.item():.4f}", "tot": f"{loss.item():.4f}"})

        scheduler.step()
        avg_fc = float(np.mean(train_fc_losses))
        avg_bc = float(np.mean(train_bc_losses))
        avg_total = float(np.mean(train_total_losses))
        val_mse, val_mae = evaluate(model, val_loader, criterion)

        logger.info(
            f"  Epoch {epoch:3d}/{epochs} | "
            f"TrainFC: {avg_fc:.6f} | TrainBC: {avg_bc:.6f} | TrainTotal: {avg_total:.6f} | "
            f"Val MSE: {val_mse:.6f} MAE: {val_mae:.6f} | "
            f"LR: {scheduler.get_last_lr()[0]:.2e}"
        )

        improved = val_mse < best_val_mse
        if improved:
            best_val_mse = val_mse
            patience_counter = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "flags": {
                    "encoding_mode": ENCODING_MODE,
                    "readout_mode": READOUT_MODE,
                    "phase_alpha": PHASE_ALPHA,
                    "use_backcast_loss": USE_BACKCAST_LOSS,
                    "use_intensity_affine": USE_INTENSITY_AFFINE,
                },
                "summary": model.optical_summary(),
                "best_val_mse": best_val_mse,
                "epoch": epoch,
            }, ckpt_path)
        else:
            patience_counter += 1

        # Always save resumable epoch state.
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "flags": {
                "encoding_mode": ENCODING_MODE,
                "readout_mode": READOUT_MODE,
                "phase_alpha": PHASE_ALPHA,
                "use_backcast_loss": USE_BACKCAST_LOSS,
                "use_intensity_affine": USE_INTENSITY_AFFINE,
            },
            "summary": model.optical_summary(),
            "best_val_mse": best_val_mse,
            "patience_counter": patience_counter,
            "epoch": epoch,
        }, last_ckpt_path)

        if patience_counter >= patience:
            logger.info(f"  Early stopping at epoch {epoch}")
            break

    # Training for this run is complete; keep the best checkpoint, remove resumable temp checkpoint.
    if os.path.exists(last_ckpt_path):
        os.remove(last_ckpt_path)

    return best_val_mse


def load_checkpoint_state(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model.load_state_dict(ckpt)
    return model


# =============================================================================
# 11. BENCHMARK LOOP
# =============================================================================

def build_model(cfg, pred_len, num_layers):
    grid_size = choose_grid_size(pred_len)
    return HAMON(
        lookback=SEQ_LEN,
        horizon=pred_len,
        channels=cfg["channels"],
        num_layers=num_layers,
        grid_size=grid_size,
        encoding_mode=ENCODING_MODE,
        readout_mode=READOUT_MODE,
        phase_alpha=PHASE_ALPHA,
        use_intensity_affine=USE_INTENSITY_AFFINE,
    ).to(device)


def run_benchmark(
    datasets=None,
    pred_lens=None,
    layer_candidates=None,
    seeds=None,
):
    datasets = datasets or list(DATASETS_CONFIG.keys())
    pred_lens = pred_lens or PRED_LENS
    layer_candidates = layer_candidates or LAYER_CANDIDATES
    seeds = seeds or SEEDS

    all_results = []
    results_path = os.path.join(EXPERIMENT_RESULTS_DIR, f"results_{EXPERIMENT_TAG}.json")
    master_logger = get_logger(f"HAMON_{EXPERIMENT_TAG}_benchmark")

    master_logger.info("=" * 90)
    master_logger.info("HAMON ULTIMATE BENCHMARK")
    master_logger.info(f"Experiment tag: {EXPERIMENT_TAG}")
    master_logger.info(f"Datasets: {datasets}")
    master_logger.info(f"Horizons: {pred_lens}")
    master_logger.info(f"Layer candidates: {layer_candidates}")
    master_logger.info(f"Seeds: {seeds}")
    master_logger.info(f"Device: {device}")
    master_logger.info(f"Save dir: {EXPERIMENT_SAVE_DIR}")
    master_logger.info(f"Flags:")
    master_logger.info(f"  ENCODING_MODE={ENCODING_MODE}")
    master_logger.info(f"  READOUT_MODE={READOUT_MODE}")
    master_logger.info(f"  PHASE_ALPHA={PHASE_ALPHA}")
    master_logger.info(f"  USE_BACKCAST_LOSS={USE_BACKCAST_LOSS}")
    master_logger.info(f"  USE_INTENSITY_AFFINE={USE_INTENSITY_AFFINE}")
    master_logger.info("=" * 90)

    for dataset_name in datasets:
        train_data, val_data, test_data, scaler, cfg = load_data(dataset_name)

        master_logger.info(f"\n{'=' * 90}")
        master_logger.info(f"DATASET: {dataset_name} | Channels: {cfg['channels']} | Batch: {cfg['batch_size']}")
        master_logger.info(f"{'=' * 90}")

        for pred_len in pred_lens:
            master_logger.info(f"\n  --- Horizon: {pred_len} ---")
            train_loader, val_loader, test_loader = make_loaders(
                train_data,
                val_data,
                test_data,
                pred_len,
                cfg["batch_size"],
            )

            # Phase 1: select best layer count using first seed.
            # If only one layer candidate is provided, skip this extra training run.
            if len(layer_candidates) == 1:
                best_layers = layer_candidates[0]
                best_val = None
                master_logger.info(f"  >> Single layer candidate: using L={best_layers} without selection run")
            else:
                selection_seed = seeds[0]
                set_seed(selection_seed)
                best_layers = None
                best_val = float("inf")

                for num_layers in layer_candidates:
                    run_name = f"{EXPERIMENT_TAG}__{dataset_name}_h{pred_len}_L{num_layers}_select"
                    logger = get_logger(run_name)

                    model = build_model(cfg, pred_len, num_layers)
                    summary = model.optical_summary()
                    logger.info(
                        f"Trying L={num_layers} | "
                        f"Params={summary['total_params']:,} | "
                        f"Optical={summary['optical_params']:,} | "
                        f"Boundary={summary['boundary_params']:,} | "
                        f"Grid={summary['grid_size']} | Span={summary['required_span']} | "
                        f"Device={summary['device_length_cm']:.0f}cm"
                    )

                    ckpt = os.path.join(EXPERIMENT_SAVE_DIR, f"{run_name}.pt")
                    val_mse = train_single(
                        model,
                        train_loader,
                        val_loader,
                        ckpt,
                        logger,
                        use_backcast_loss=USE_BACKCAST_LOSS,
                    )

                    master_logger.info(f"    L={num_layers:2d} | Val MSE: {val_mse:.6f}")
                    if val_mse < best_val:
                        best_val = val_mse
                        best_layers = num_layers

                master_logger.info(f"  >> Best layers: {best_layers} (Val MSE: {best_val:.6f})")

            # Phase 2: train best layer count over seeds.
            seed_mses, seed_maes = [], []

            for seed in seeds:
                set_seed(seed)
                run_name = f"{EXPERIMENT_TAG}__{dataset_name}_h{pred_len}_L{best_layers}_s{seed}"
                logger = get_logger(run_name)
                logger.info(f"Seed {seed} | L={best_layers} | H={pred_len} | Dataset={dataset_name}")

                model = build_model(cfg, pred_len, best_layers)
                summary = model.optical_summary()
                logger.info(json.dumps(summary, indent=2))

                ckpt = os.path.join(EXPERIMENT_SAVE_DIR, f"{run_name}.pt")
                train_single(
                    model,
                    train_loader,
                    val_loader,
                    ckpt,
                    logger,
                    use_backcast_loss=USE_BACKCAST_LOSS,
                )

                model = build_model(cfg, pred_len, best_layers)
                model = load_checkpoint_state(model, ckpt)
                model.eval()

                test_mse, test_mae = evaluate(model, test_loader, nn.MSELoss())
                seed_mses.append(test_mse)
                seed_maes.append(test_mae)

                logger.info(f"Seed {seed} TEST | MSE: {test_mse:.6f} MAE: {test_mae:.6f}")

            avg_mse = float(np.mean(seed_mses))
            std_mse = float(np.std(seed_mses))
            avg_mae = float(np.mean(seed_maes))
            std_mae = float(np.std(seed_maes))

            final_summary = build_model(cfg, pred_len, best_layers).optical_summary()
            result = {
                "experiment_tag": EXPERIMENT_TAG,
                "dataset": dataset_name,
                "horizon": pred_len,
                "layers": best_layers,
                "params": final_summary["total_params"],
                "optical_params": final_summary["optical_params"],
                "boundary_params": final_summary["boundary_params"],
                "device_cm": final_summary["device_length_cm"],
                "grid_size": final_summary["grid_size"],
                "required_span": final_summary["required_span"],
                "encoding_mode": ENCODING_MODE,
                "readout_mode": READOUT_MODE,
                "phase_alpha": PHASE_ALPHA,
                "use_backcast_loss": USE_BACKCAST_LOSS,
                "use_intensity_affine": USE_INTENSITY_AFFINE,
                "mse_mean": avg_mse,
                "mse_std": std_mse,
                "mae_mean": avg_mae,
                "mae_std": std_mae,
                "mse_per_seed": [float(m) for m in seed_mses],
                "mae_per_seed": [float(m) for m in seed_maes],
            }
            all_results.append(result)

            master_logger.info(
                f"RESULT | {dataset_name} H={pred_len} L={best_layers} | "
                f"MSE: {avg_mse:.6f}±{std_mse:.6f} | "
                f"MAE: {avg_mae:.6f}±{std_mae:.6f} | "
                f"Params: {final_summary['total_params']:,} | Grid: {final_summary['grid_size']}"
            )

            with open(results_path, "w") as f:
                json.dump(all_results, f, indent=2)

    master_logger.info(f"\n\n{'=' * 100}")
    master_logger.info("HAMON ULTIMATE — FULL RESULTS")
    master_logger.info(f"{'=' * 100}")
    master_logger.info(
        f"{'Experiment':<44} {'Dataset':<12} {'H':>4} {'L':>3} {'Grid':>5} "
        f"{'Params':>8} {'MSE':>15} {'MAE':>15}"
    )
    master_logger.info("-" * 100)

    for r in all_results:
        master_logger.info(
            f"{r['experiment_tag']:<44} {r['dataset']:<12} {r['horizon']:>4d} "
            f"{r['layers']:>3d} {r['grid_size']:>5d} {r['params']:>8d} "
            f"{r['mse_mean']:>8.4f}±{r['mse_std']:.4f} "
            f"{r['mae_mean']:>8.4f}±{r['mae_std']:.4f}"
        )

    master_logger.info(f"{'=' * 100}")
    master_logger.info(f"Results saved to {results_path}")
    return all_results


# =============================================================================
# 12. QUICK PRESETS
# =============================================================================

"""
Copy one of these flag blocks near the top before running.

# A) Existing baseline
ENCODING_MODE = "amplitude"
READOUT_MODE = "coherent"
PHASE_ALPHA = 0.3
USE_BACKCAST_LOSS = True
USE_INTENSITY_AFFINE = False

# B) Main invention candidate: phase encoding + coherent readout
ENCODING_MODE = "phase"
READOUT_MODE = "coherent"
PHASE_ALPHA = 0.3
USE_BACKCAST_LOSS = True
USE_INTENSITY_AFFINE = False

# C) Phase encoding + intensity readout + boundary affine calibration
ENCODING_MODE = "phase"
READOUT_MODE = "intensity"
PHASE_ALPHA = 0.3
USE_BACKCAST_LOSS = False
USE_INTENSITY_AFFINE = True

# D) Phase encoding + differential intensity readout
ENCODING_MODE = "phase"
READOUT_MODE = "differential_intensity"
PHASE_ALPHA = 0.3
USE_BACKCAST_LOSS = False
USE_INTENSITY_AFFINE = True
"""


# =============================================================================
# 13. ENTRY POINT
# =============================================================================

# Smaller sanity run first. Recommended before full benchmark.
RUN_SANITY_CHECK = False

if RUN_SANITY_CHECK:
    results = run_benchmark(
        datasets=["ettm2"],
        pred_lens=[96],
        layer_candidates=[16],
        seeds=[1],
    )
else:
    # Phase encoding + differential intensity readout,
    # no Electricity/Traffic, L=16, three seeds.
    results = run_benchmark(
        datasets=["etth1", "etth2", "ettm1", "ettm2", "weather"],
        pred_lens=[96, 192, 336, 720],
        layer_candidates=[16],
        seeds=[1, 2, 3],
    )

print("Done.")
print(json.dumps(results, indent=2))


In [ ]:
import os
import json
import glob
import pandas as pd
from IPython.display import display

BASE_DIR = "/content/drive/MyDrive/HAMON_Ultimate"
RESULTS_DIR = os.path.join(BASE_DIR, "results")

json_files = glob.glob(
    os.path.join(RESULTS_DIR, "**", "results_*.json"),
    recursive=True
)

if not json_files:
    print("Result JSON bulunamadı.")
else:
    latest = max(json_files, key=os.path.getmtime)

    print("Latest result file:")
    print(latest)

    with open(latest, "r") as f:
        rows = json.load(f)

    df = pd.DataFrame(rows)

    if df.empty:
        print("Result dosyası boş.")
    else:
        sort_cols = [c for c in ["dataset", "horizon", "layers"] if c in df.columns]
        df = df.sort_values(sort_cols).reset_index(drop=True)

        cols = [
            "experiment_tag",
            "dataset",
            "horizon",
            "layers",
            "grid_size",
            "params",
            "mse_mean",
            "mse_std",
            "mae_mean",
            "mae_std",
            "mse_per_seed",
            "mae_per_seed",
        ]

        display(df[[c for c in cols if c in df.columns]])

        print("\nCompact:")
        print("=" * 130)

        for _, r in df.iterrows():
            exp = str(r.get("experiment_tag", ""))
            dataset = str(r.get("dataset", ""))
            h = int(r.get("horizon", 0))
            L = int(r.get("layers", 0))
            grid = int(r.get("grid_size", 0)) if pd.notnull(r.get("grid_size", None)) else 0
            params = int(r.get("params", 0)) if pd.notnull(r.get("params", None)) else 0

            mse_mean = float(r.get("mse_mean", float("nan")))
            mse_std = float(r.get("mse_std", float("nan")))
            mae_mean = float(r.get("mae_mean", float("nan")))
            mae_std = float(r.get("mae_std", float("nan")))

            print(
                f"{exp:<78} "
                f"{dataset:<10} "
                f"H={h:<3} "
                f"L={L:<2} "
                f"G={grid:<4} "
                f"P={params:<7} "
                f"MSE={mse_mean:.4f}±{mse_std:.4f} "
                f"MAE={mae_mean:.4f}±{mae_std:.4f}"
            )

        print("=" * 130)